# CTI Dataset Builder — Hugging Face

Research: Cyber-Threat Classification and Emerging Threat Detection.

This notebook downloads the selected datasets, inspects their real schemas, standardizes CVE IDs, preserves source-specific labels, checks overlap, and exports a combined source-record dataset.

**Keys:** `uid` is unique per source row; `cve_id` is the shared matching key and may occur in both datasets. Missing labels are not negative labels. `threat_category` is left blank until you define a documented annotation rule.

## 1. Install dependencies

In [2]:
%pip install -q datasets pandas pyarrow scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from datasets import load_dataset
import pandas as pd
import numpy as np
import re
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
OUTPUT_DIR = Path("../dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Define sources

In [4]:
CIRCL_DATASET = "CIRCL/vulnerability-attack-techniques"
THREATCLUSTER_DATASET = "threatcluster/cve-exploitation-signals"

print("CIRCL: https://huggingface.co/datasets/" + CIRCL_DATASET)
print("ThreatCluster: https://huggingface.co/datasets/" + THREATCLUSTER_DATASET)


CIRCL: https://huggingface.co/datasets/CIRCL/vulnerability-attack-techniques
ThreatCluster: https://huggingface.co/datasets/threatcluster/cve-exploitation-signals


## 3. Download datasets from Hugging Face
Run this cell. If Hugging Face changes a repository schema or split, the error/output will help us adjust the loader.

In [5]:
circl_ds = load_dataset(CIRCL_DATASET)
threatcluster_ds = load_dataset(THREATCLUSTER_DATASET)
print("CIRCL splits:", circl_ds)
print("ThreatCluster splits:", threatcluster_ds)


CIRCL splits: DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'description', 'exploitation_techniques', 'primary_impact', 'secondary_impact', 'techniques', 'techniques_derived', 'label_sources', 'attack_version', 'cvss_vector', 'cvss_version', 'cwes', 'affected_products', 'cpes', 'cwes_predicted'],
        num_rows: 1086
    })
    test: Dataset({
        features: ['id', 'title', 'description', 'exploitation_techniques', 'primary_impact', 'secondary_impact', 'techniques', 'techniques_derived', 'label_sources', 'attack_version', 'cvss_vector', 'cvss_version', 'cwes', 'affected_products', 'cpes', 'cwes_predicted'],
        num_rows: 121
    })
})
ThreatCluster splits: DatasetDict({
    train: Dataset({
        features: ['cve_id', 'description', 'published_date', 'cvss_v3_score', 'cvss_v3_severity', 'cvss_v3_vector', 'cwe_ids', 'affected_vendors', 'affected_products', 'in_kev', 'kev_added_date', 'kev_due_date', 'ransomware_use', 'has_exploit', 'exploit_count'],
    

## 4. Select a split and convert to pandas

In [6]:
def choose_split(ds):
    name = "train" if "train" in ds else next(iter(ds.keys()))
    print(f"Using split {name!r}: {ds[name].num_rows:,} rows")
    return ds[name].to_pandas()

circl_raw = choose_split(circl_ds)
threatcluster_raw = choose_split(threatcluster_ds)
print("CIRCL shape:", circl_raw.shape)
print("ThreatCluster shape:", threatcluster_raw.shape)


Using split 'train': 1,086 rows
Using split 'train': 60,879 rows
CIRCL shape: (1086, 16)
ThreatCluster shape: (60879, 15)


## 5. Inspect schemas and missing values

In [7]:
def schema_report(df, label):
    report = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "missing_count": [int(df[c].isna().sum()) for c in df.columns],
        "missing_percent": [round(float(df[c].isna().mean() * 100), 2) for c in df.columns],
        "example": [next((str(v)[:160] for v in df[c].dropna().head(5)), "") for c in df.columns]
    })
    print(label)
    display(report)
    return report

circl_schema = schema_report(circl_raw, "CIRCL")
tc_schema = schema_report(threatcluster_raw, "ThreatCluster")


CIRCL


,column,dtype,missing_count,missing_percent,example
0,id,object,0,0.0,CVE-2023-36851
1,title,object,0,0.0,Junos OS: SRX Series: A vulnerability in J-Web allows an unauthenticated attacker to upload and download arbitrary f...
2,description,object,0,0.0,A Missing Authentication for Critical Function vulnerability in Juniper Networks Junos OS on SRX Series allows an un...
3,exploitation_techniques,object,0,0.0,['T1190']
4,primary_impact,object,0,0.0,['T1059']
5,secondary_impact,object,0,0.0,[]
6,techniques,object,0,0.0,['T1059' 'T1190']
7,techniques_derived,object,0,0.0,['T1014' 'T1027.009' 'T1037' 'T1040' 'T1080' 'T1134' 'T1185' 'T1505.003'\n 'T1505.005' 'T1542.003' 'T1543' 'T1543.00...
8,label_sources,object,0,0.0,['ctid_kev']
9,attack_version,object,0,0.0,19.1


ThreatCluster


,column,dtype,missing_count,missing_percent,example
0,cve_id,object,0,0.00,CVE-2026-86151
1,description,object,0,0.00,A vulnerability was detected in Tenda CP3 27.5.57.101. The affected element is the function sub_2F77E8 of the file A...
2,published_date,datetime64[s],0,0.00,2026-09-06 00:00:00
3,cvss_v3_score,object,6551,10.76,9.1
4,cvss_v3_severity,object,6551,10.76,CRITICAL
5,cvss_v3_vector,object,6551,10.76,CVSS:3.1/AV:N/AC:L/PR:H/UI:N/S:C/C:H/I:H/A:H
6,cwe_ids,object,5411,8.89,['CWE-77' 'CWE-78']
7,affected_vendors,object,19228,31.58,['google']
8,affected_products,object,19225,31.58,['chrome']
9,in_kev,bool,0,0.00,False


## 6. Standardize CVE identifiers and inspect overlap

In [8]:
def normalize_cve(value):
    if pd.isna(value):
        return pd.NA
    match = re.search(r"CVE-\d{4}-\d{4,}", str(value).strip().upper())
    return match.group(0) if match else pd.NA

if "id" not in circl_raw.columns:
    raise KeyError(f"CIRCL expected 'id'; actual columns: {circl_raw.columns.tolist()}")
if "cve_id" not in threatcluster_raw.columns:
    raise KeyError(f"ThreatCluster expected 'cve_id'; actual columns: {threatcluster_raw.columns.tolist()}")

circl = circl_raw.copy()
tc = threatcluster_raw.copy()
circl["cve_id"] = circl["id"].map(normalize_cve)
tc["cve_id"] = tc["cve_id"].map(normalize_cve)

circl_ids = set(circl["cve_id"].dropna())
tc_ids = set(tc["cve_id"].dropna())
overlap_ids = circl_ids & tc_ids

print("CIRCL rows / unique CVEs:", len(circl), len(circl_ids))
print("ThreatCluster rows / unique CVEs:", len(tc), len(tc_ids))
print("CVEs in both sources:", len(overlap_ids))


CIRCL rows / unique CVEs: 1086 1086
ThreatCluster rows / unique CVEs: 60879 60879
CVEs in both sources: 0


## 7. Build a harmonized combined dataset
This is a row-wise union of source records, not a one-row-per-CVE merge. CVEs found in both sources remain as separate rows so source annotations are not lost.

In [9]:
circl["source"] = "circl"
tc["source"] = "threatcluster"

def series_or_na(df, name):
    return df[name] if name in df.columns else pd.Series(pd.NA, index=df.index)

def harmonize(df, source):
    out = pd.DataFrame(index=df.index)
    out["cve_id"] = series_or_na(df, "cve_id")
    out["source"] = source
    out["description"] = series_or_na(df, "description")
    out["published_date"] = series_or_na(df, "published_date")
    out["cvss_v3_score"] = series_or_na(df, "cvss_v3_score")
    out["cvss_v3_severity"] = series_or_na(df, "cvss_v3_severity")
    out["cvss_v3_vector"] = series_or_na(df, "cvss_v3_vector")
    out["cwe_ids"] = series_or_na(df, "cwe_ids") if "cwe_ids" in df.columns else series_or_na(df, "cwes")
    out["affected_vendors"] = series_or_na(df, "affected_vendors")
    out["affected_products"] = series_or_na(df, "affected_products")
    out["in_kev"] = series_or_na(df, "in_kev")
    out["has_exploit"] = series_or_na(df, "has_exploit")
    out["exploit_count"] = series_or_na(df, "exploit_count")
    out["ransomware_use"] = series_or_na(df, "ransomware_use")
    out["kev_added_date"] = series_or_na(df, "kev_added_date")
    out["kev_due_date"] = series_or_na(df, "kev_due_date")
    out["mitre_techniques"] = series_or_na(df, "exploitation_techniques")
    if source == "circl" and "exploitation_techniques" not in df.columns:
        out["mitre_techniques"] = series_or_na(df, "techniques")
    out["primary_impact"] = series_or_na(df, "primary_impact")
    out["secondary_impact"] = series_or_na(df, "secondary_impact")
    out["techniques_derived"] = series_or_na(df, "techniques_derived")
    out["label_sources"] = series_or_na(df, "label_sources")
    out["threat_category"] = pd.NA  # annotate only with documented evidence/rules
    return out.reset_index(drop=True)

combined = pd.concat([harmonize(circl, "circl"), harmonize(tc, "threatcluster")], ignore_index=True)
combined["cve_id"] = combined["cve_id"].map(normalize_cve)
combined["_row_num"] = combined.groupby(["source", "cve_id"], dropna=False).cumcount() + 1
combined["uid"] = combined["source"] + "_" + combined["cve_id"].fillna("UNKNOWN") + "_" + combined["_row_num"].astype(str)
combined["uid"] = combined["uid"].str.replace(r"_1$", "", regex=True)
combined = combined.drop(columns=["_row_num"])

print("Combined shape:", combined.shape)
print("Unique uid:", combined["uid"].nunique())
print("Duplicate uid rows:", int(combined["uid"].duplicated().sum()))
display(combined.head(10))


C:\Users\Hp\AppData\Local\Temp\ipykernel_15992\144879527.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([harmonize(circl, "circl"), harmonize(tc, "threatcluster")], ignore_index=True)
C:\Users\Hp\AppData\Local\Temp\ipykernel_15992\144879527.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([harmonize(circl, "circl"), harmonize(tc, "threatcluster")], ignore_index=True)


Combined shape: (61965, 23)
Unique uid: 61965
Duplicate uid rows: 0


,cve_id,source,description,published_date,cvss_v3_score,cvss_v3_severity,cvss_v3_vector,cwe_ids,affected_vendors,affected_products,in_kev,has_exploit,exploit_count,ransomware_use,kev_added_date,kev_due_date,mitre_techniques,primary_impact,secondary_impact,techniques_derived,label_sources,threat_category,uid
0,CVE-2023-36851,circl,A Missing Authentication for Critical Function vulnerability in Juniper Networks Junos OS on SRX Series allows an un...,NaT,<NA>,<NA>,<NA>,[CWE-306 Missing Authentication for Critical Function],<NA>,[Juniper Networks Junos OS],<NA>,<NA>,<NA>,NaN,NaT,NaT,[T1190],[T1059],[],"[T1014, T1027.009, T1037, T1040, T1080, T1134, T1185, T1505.003, T1505.005, T1542.003, T1543, T1543.001, T1543.003, ...",[ctid_kev],<NA>,circl_CVE-2023-36851
1,CVE-2019-1087,circl,"An elevation of privilege exists in Windows Audio Service, aka 'Windows Audio Service Elevation of Privilege Vulnera...",NaT,<NA>,<NA>,<NA>,[Elevation of Privilege],<NA>,"[Microsoft Windows, Microsoft Windows Server, Microsoft Windows 10 Version 1903 for 32-bit Systems, Microsoft Window...",<NA>,<NA>,<NA>,NaN,NaT,NaT,[],[T1068],[],[],[ctid_cve],<NA>,circl_CVE-2019-1087
2,CVE-2020-3322,circl,A vulnerability in Cisco Webex Network Recording Player and Cisco Webex Player for Microsoft Windows could allow an ...,NaT,<NA>,<NA>,<NA>,[CWE-20 Improper Input Validation],<NA>,"[Cisco Cisco Webex Network Recording Player, Cisco Cisco Webex Player for Microsoft Windows]",<NA>,<NA>,<NA>,NaN,NaT,NaT,"[T1204.002, T1566]",[T1499.004],[],"[T1027, T1036.001, T1539, T1553.002, T1562.003, T1574.006, T1574.007]",[ctid_cve],<NA>,circl_CVE-2020-3322
3,CVE-2023-42793,circl,In JetBrains TeamCity before 2023.05.4 authentication bypass leading to RCE on TeamCity Server was possible,NaT,<NA>,<NA>,<NA>,[CWE-288],<NA>,[JetBrains TeamCity],<NA>,<NA>,<NA>,NaN,NaT,NaT,[T1190],[T1059.003],[],"[T1014, T1027.009, T1037, T1040, T1080, T1134, T1185, T1505.003, T1505.005, T1542.003, T1543, T1543.001, T1543.003, ...",[ctid_kev],<NA>,circl_CVE-2023-42793
4,CVE-2019-0911,circl,A remote code execution vulnerability exists in the way the scripting engine handles objects in memory in Microsoft ...,NaT,<NA>,<NA>,<NA>,[Remote Code Execution],<NA>,"[Microsoft Internet Explorer 11, Microsoft Internet Explorer 11 on Windows Server 2012, Microsoft Internet Explorer ...",<NA>,<NA>,<NA>,NaN,NaT,NaT,"[T1189, T1204.002]","[T1499.004, T1574]",[],"[T1134, T1134.001, T1134.002, T1134.003, T1528, T1539, T1550.004, T1606]",[ctid_cve],<NA>,circl_CVE-2019-0911
5,CVE-2020-1027,circl,"An elevation of privilege vulnerability exists in the way that the Windows Kernel handles objects in memory, aka 'Wi...",NaT,<NA>,<NA>,<NA>,"[Elevation of Privilege, CWE-787 Out-of-bounds Write]",<NA>,"[Microsoft Windows, Microsoft Windows Server, Microsoft Windows 10 Version 1909 for 32-bit Systems, Microsoft Window...",<NA>,<NA>,<NA>,NaN,NaT,NaT,[],[],[],"[T1134, T1134.001, T1134.002, T1134.003, T1528, T1539, T1550.004, T1606]",[ctid_cve],<NA>,circl_CVE-2020-1027
6,CVE-2017-8464,circl,"Windows Shell in Microsoft Windows Server 2008 SP2 and R2 SP1, Windows 7 SP1, Windows 8, Windows 8.1, Windows Server...",NaT,<NA>,<NA>,<NA>,"[Remote Code Execution, CWE-noinfo Not enough information]",<NA>,[Microsoft Corporation Windows Shell],<NA>,<NA>,<NA>,NaN,NaT,NaT,[],[],[],[],[ctid_cve],<NA>,circl_CVE-2017-8464
7,CVE-2020-11059,circl,"In AEgir greater than or equal to 21.7.0 and less than 21.10.1, aegir publish and aegir build may leak secrets from ...",NaT,<NA>,<NA>,<NA>,[CWE-200: Exposure of Sensitive Information to an Unauthorized Actor],<NA>,[IPFS AEgir],<NA>,<NA>,<NA>,NaN,NaT,NaT,[],[T1005],[],"[T1007, T1016, T1018, T1033, T1036.005, T1046, T1049, T1057, T1069, T1082, T1083, T1087, T1111, T1120, T1124, T1134,...",[ctid_cve],<NA>,circl_CVE-2020-11059
8,CVE-2018-15801,circl,Spring Security versions 5.1.x prior to 5.1.2 contain an authorization bypass vulnerability during JWT issuer valida...,NaT,<NA>,<NA>,<NA>,[Business Logic Errors],<NA>,[Sprin

## 8. Data-quality summary

In [10]:
quality = pd.DataFrame([
    {"check": "Combined rows", "value": len(combined)},
    {"check": "Unique uid", "value": combined["uid"].nunique()},
    {"check": "Duplicate uid rows", "value": int(combined["uid"].duplicated().sum())},
    {"check": "Missing CVE IDs", "value": int(combined["cve_id"].isna().sum())},
    {"check": "Missing descriptions", "value": int(combined["description"].isna().sum()) if "description" in combined else len(combined)},
    {"check": "Unique CVEs", "value": combined["cve_id"].nunique(dropna=True)},
    {"check": "CVE IDs in both sources", "value": len(overlap_ids)}
])
display(quality)
display(combined.groupby("source").agg(
    rows=("uid", "size"),
    unique_cves=("cve_id", "nunique")
).reset_index())


,check,value
0,Combined rows,61965
1,Unique uid,61965
2,Duplicate uid rows,0
3,Missing CVE IDs,0
4,Missing descriptions,0
5,Unique CVEs,61965
6,CVE IDs in both sources,0


,source,rows,unique_cves
0,circl,1086,1086
1,threatcluster,60879,60879


## 9. Review CVEs that occur in both sources

In [11]:
overlap_review = combined[combined["cve_id"].isin(overlap_ids)].sort_values(["cve_id", "source"])
display(overlap_review[["cve_id", "source", "description", "cwe_ids", "affected_products", "mitre_techniques"]].head(30))


,cve_id,source,description,cwe_ids,affected_products,mitre_techniques


## 10. Export CSV and Parquet
Nested list fields are converted to readable strings in the CSV export. The original source dataframes remain available in this notebook.

In [12]:
def csv_safe(value):
    if value is None or value is pd.NA:
        return ""
    if isinstance(value, (list, tuple, set, np.ndarray)):
        return "; ".join(str(x) for x in value if x is not None)
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value)

csv_export = combined.copy()
for col in csv_export.columns:
    csv_export[col] = csv_export[col].map(csv_safe)

csv_path = OUTPUT_DIR / "SMAH-CTI67.csv"
parquet_path = OUTPUT_DIR / "SMAH-CTI67.parquet"
csv_export.to_csv(csv_path, index=False, encoding="utf-8-sig")
combined.to_parquet(parquet_path, index=False)

print("CSV:", csv_path.resolve())
print("Parquet:", parquet_path.resolve())
print("Rows exported:", len(combined))


CSV: D:\Projects\done\thesis\cyber_threat_classification_n_emerging_threat_detection\dataset\SMAH-CTI67.csv
Parquet: D:\Projects\done\thesis\cyber_threat_classification_n_emerging_threat_detection\dataset\SMAH-CTI67.parquet
Rows exported: 61965


## 11. Before model training

- Define `threat_category` from verified labels or a documented annotation procedure; these two schemas do not guarantee Malware/Phishing/Botnet/DDoS/Spam labels for every CVE.
- Keep publication date and KEV-added date separate.
- Do not treat missing technique labels as negative labels.
- If `relevant` is derived from `in_kev`, `has_exploit`, `exploit_count`, `ransomware_use`, or description keywords, exclude those label-generating signals from model inputs for that task to avoid target leakage.
- Keep all rows for the same `cve_id` in one train/test split to prevent leakage across sources.
- Use chronological evaluation when making emerging-threat prediction claims.
- Review each Hugging Face dataset card for the current schema and license/usage terms before publishing results.